# 🧾 Carbon Crunch – AI-OCR Receipt Processor

Run each cell top to bottom.

In [1]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install -q easyocr opencv-python-headless numpy Pillow
print('✅ Dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 22.3 MB/s eta 0:00:00
✅ Dependencies installed


In [2]:

from google.colab import files
import os

print('📂 A file picker will open — select your receipts ZIP file...')
uploaded = files.upload()   # ← pick your zip here

zip_name = list(uploaded.keys())[0]
print(f'\n✅ Uploaded: {zip_name}')

# Unzip into /content/receipts/
os.makedirs('/content/receipts', exist_ok=True)
!unzip -q "/content/{zip_name}" -d /content/receipts/

# Handle nested folder (some zips extract into a subfolder)
subdirs = [d for d in os.listdir('/content/receipts') if os.path.isdir(f'/content/receipts/{d}')]
if subdirs:
    RECEIPTS_FOLDER = f'/content/receipts/{subdirs[0]}'
else:
    RECEIPTS_FOLDER = '/content/receipts'

# Count images
exts = {'.jpg','.jpeg','.png','.bmp','.tiff','.webp'}
imgs = [f for f in os.listdir(RECEIPTS_FOLDER) if os.path.splitext(f)[1].lower() in exts]
print(f'✅ Found {len(imgs)} image(s) in: {RECEIPTS_FOLDER}')
for f in imgs[:10]: print(f'   • {f}')
if len(imgs) > 10: print(f'   … and {len(imgs)-10} more')

📂 A file picker will open — select your receipts ZIP file...


Saving AI-OCR dataset-20260427T044938Z-3-001.zip to AI-OCR dataset-20260427T044938Z-3-001.zip

✅ Uploaded: AI-OCR dataset-20260427T044938Z-3-001.zip
✅ Found 371 image(s) in: /content/receipts/AI-OCR dataset
   • X51005361912.jpg
   • X51005745214.jpg
   • X51005447848.jpg
   • X51005757349.jpg
   • X00016469669.jpg
   • X51005719882.jpg
   • X51005676537.jpg
   • X51005717526.jpg
   • X51005742068.jpg
   • X51005587261.jpg
   … and 361 more


In [3]:

import os
os.makedirs('/content/ai_ocr/src', exist_ok=True)
os.makedirs('/content/ai_ocr/outputs', exist_ok=True)

ocr_pipeline_src = '''
import cv2, numpy as np, re
from pathlib import Path
from dataclasses import dataclass

try:
    import easyocr
    EASYOCR_AVAILABLE = True
except ImportError:
    EASYOCR_AVAILABLE = False

try:
    import pytesseract
    from PIL import Image
    TESSERACT_AVAILABLE = True
except ImportError:
    TESSERACT_AVAILABLE = False

LOW_CONF_THRESHOLD = 0.70

@dataclass
class ConfidentField:
    value: str
    confidence: float
    flagged: bool = False
    def to_dict(self):
        return {"value": self.value, "confidence": round(self.confidence,4), "flagged": self.flagged}

@dataclass
class ReceiptItem:
    name: str; price: str; confidence: float
    def to_dict(self):
        return {"name": self.name, "price": self.price, "confidence": round(self.confidence,4)}

@dataclass
class ReceiptResult:
    file: str; store_name: ConfidentField; date: ConfidentField
    items: list; total_amount: ConfidentField; raw_text: str
    ocr_engine: str; preprocessing_steps: list; overall_confidence: float = 0.0
    def to_dict(self):
        return {"file": self.file, "store_name": self.store_name.to_dict(),
                "date": self.date.to_dict(), "items": [i.to_dict() for i in self.items],
                "total_amount": self.total_amount.to_dict(), "raw_text": self.raw_text,
                "ocr_engine": self.ocr_engine, "preprocessing_steps": self.preprocessing_steps,
                "overall_confidence": round(self.overall_confidence,4)}

class ImagePreprocessor:
    @staticmethod
    def _deskew(g):
        edges = cv2.Canny(g, 50, 150, apertureSize=3)
        lines = cv2.HoughLinesP(edges,1,np.pi/180,80,minLineLength=50,maxLineGap=10)
        if lines is None: return g, 0.0
        angles = [np.degrees(np.arctan2(l[0][3]-l[0][1],l[0][2]-l[0][0])) for l in lines if l[0][2]!=l[0][0]]
        if not angles: return g, 0.0
        a = np.median(angles)
        if abs(a)<0.5: return g, a
        h,w=g.shape; M=cv2.getRotationMatrix2D((w/2,h/2),a,1.0)
        return cv2.warpAffine(g,M,(w,h),flags=cv2.INTER_CUBIC,borderMode=cv2.BORDER_REPLICATE), a
    def preprocess(self, path):
        steps=[]; img=cv2.imread(path)
        if img is None: raise ValueError(f"Cannot read: {path}")
        h,w=img.shape[:2]
        if max(h,w)<1000:
            s=1000/max(h,w); img=cv2.resize(img,None,fx=s,fy=s,interpolation=cv2.INTER_CUBIC); steps.append("upscale")
        gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY); steps.append("grayscale")
        gray=cv2.fastNlMeansDenoising(gray,h=10); steps.append("denoise")
        clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)); gray=clahe.apply(gray); steps.append("clahe")
        gray,angle=self._deskew(gray)
        if abs(angle)>0.5: steps.append(f"deskew({angle:.1f}deg)")
        binary=cv2.adaptiveThreshold(gray,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,31,11); steps.append("binarize")
        return binary, steps

class OCREngine:
    def __init__(self, engine="auto"):
        if engine=="auto": engine="easyocr" if EASYOCR_AVAILABLE else "tesseract"
        self.engine=engine
        if engine=="easyocr": self._reader=easyocr.Reader(["en"],gpu=False,verbose=False)
    def run(self, img):
        if self.engine=="easyocr":
            return [{"text":t.strip(),"confidence":float(c),"bbox":b} for b,t,c in self._reader.readtext(img,detail=1,paragraph=False)]
        pil=Image.fromarray(img); d=pytesseract.image_to_data(pil,output_type=pytesseract.Output.DICT,config="--psm 6")
        out=[]
        for i,t in enumerate(d["text"]):
            t=t.strip()
            if not t: continue
            c=max(0.0,float(d["conf"][i]))/100; x,y,w,h=d["left"][i],d["top"][i],d["width"][i],d["height"][i]
            out.append({"text":t,"confidence":c,"bbox":[[x,y],[x+w,y],[x+w,y+h],[x,y+h]]})
        return out

DATE_PATS=[r"\\b\\d{1,2}[/\\-\\.]\\d{1,2}[/\\-\\.]\\d{2,4}\\b",r"\\b\\d{4}[/\\-\\.]\\d{1,2}[/\\-\\.]\\d{1,2}\\b",
           r"\\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\\.?\\s+\\d{1,2},?\\s+\\d{4}\\b"]
CUR=r"[$\u20ac\u00a3\u20b9\u00a5]?\\s*\\d{1,6}(?:[,\\.]\\d{2,3})*(?:[\\.]\\d{2})?"
TOTAL_KW={"total","amount","grand total","sum","balance","due","subtotal","net","payable","paid"}

class FieldExtractor:
    @staticmethod
    def _y(w): b=w["bbox"]; return b[0][1] if isinstance(b[0],(list,tuple)) else b[1]
    def _lines(self,words,tol=12):
        if not words: return []
        s=sorted(words,key=self._y); lines=[[s[0]]]
        for w in s[1:]:
            if abs(self._y(w)-self._y(lines[-1][-1]))<=tol: lines[-1].append(w)
            else: lines.append([w])
        return lines
    @staticmethod
    def _fc(ocr,pv,kb): return min(1.0, ocr*0.6 + (0.25 if pv else 0) + kb)
    def extract_all(self,words):
        full=" ".join(w["text"] for w in words); lines=self._lines(words)
        cands=[]
        for ln in lines[:4]:
            t=" ".join(w["text"] for w in ln).strip()
            if not t or re.fullmatch(r"[\\d\\s/\\-\\.\\$\u20ac\u00a3\u20b9,]+",t): continue
            cands.append((t,float(np.mean([w["confidence"] for w in ln]))))
        if cands:
            best=max(cands[:3],key=lambda x:(len(x[0]),x[1]))
            sc=self._fc(best[1],True,0); store=ConfidentField(best[0],sc,sc<LOW_CONF_THRESHOLD)
        else:
            store=ConfidentField("UNKNOWN",0.0,True)
        date=ConfidentField("NOT FOUND",0.1,True)
        for pat in DATE_PATS:
            m=re.search(pat,full,re.IGNORECASE)
            if m:
                wc=[w["confidence"] for w in words if w["text"] in m.group(0)]
                oc=float(np.mean(wc)) if wc else 0.75
                dc=self._fc(oc,True,0.05); date=ConfidentField(m.group(0),dc,dc<LOW_CONF_THRESHOLD); break
        total=ConfidentField("NOT FOUND",0.1,True); best_tc=0
        for ln in lines:
            lt=" ".join(w["text"] for w in ln).lower()
            kw=any(k in lt for k in TOTAL_KW)
            amts=re.findall(CUR," ".join(w["text"] for w in ln))
            if not amts: continue
            oc=float(np.mean([w["confidence"] for w in ln]))
            tc=self._fc(oc,True,0.15 if kw else 0)
            if tc>best_tc: best_tc=tc; total=ConfidentField(amts[-1].strip(),tc,tc<LOW_CONF_THRESHOLD)
        items=[]
        for ln in lines:
            full_ln=" ".join(w["text"] for w in ln)
            amts=re.findall(CUR,full_ln)
            if not amts or any(k in full_ln.lower() for k in TOTAL_KW): continue
            p=amts[-1].strip(); n=full_ln[:full_ln.rfind(p)].strip()
            if len(n)<2: continue
            oc=float(np.mean([w["confidence"] for w in ln]))
            items.append(ReceiptItem(n,p,round(oc,4)))
        return {"store_name":store,"date":date,"total_amount":total,"items":items,"raw_text":full}

class ReceiptOCRPipeline:
    def __init__(self,engine="auto"):
        self.pre=ImagePreprocessor(); self.ocr=OCREngine(engine); self.ext=FieldExtractor()
    def process_image(self,path):
        fname=Path(path).name
        try: img,steps=self.pre.preprocess(path)
        except Exception as e: return self._err(fname,str(e))
        try: words=self.ocr.run(img)
        except Exception as e: return self._err(fname,str(e))
        if not words: return self._err(fname,"No text detected")
        ex=self.ext.extract_all(words)
        r=ReceiptResult(fname,ex["store_name"],ex["date"],ex["items"],ex["total_amount"],ex["raw_text"],self.ocr.engine,steps)
        ic=float(np.mean([i.confidence for i in r.items])) if r.items else 0.5
        r.overall_confidence=round(0.3*r.store_name.confidence+0.3*r.total_amount.confidence+0.2*r.date.confidence+0.2*ic,4)
        return r
    def process_directory(self,d):
        exts={\'.jpg\',\'.jpeg\',\'.png\',\'.bmp\',\'.tiff\',\'.tif\',\'.webp\'}
        paths=[p for p in Path(d).iterdir() if p.suffix.lower() in exts]
        results=[]
        for p in sorted(paths):
            print(f"  Processing {p.name}...")
            results.append(self.process_image(str(p)))
        return results
    @staticmethod
    def _err(f,msg):
        e=ConfidentField(f"ERROR:{msg}",0.0,True)
        return ReceiptResult(f,e,e,[],e,"","none",[],0.0)
'''

summary_src = '''
import re
from collections import defaultdict
CUR=re.compile(r"[$\u20ac\u00a3\u20b9\u00a5]?\\s*([\\d,]+(?:\\.\\d{1,2})?)")
def _parse(raw):
    if not raw or raw in ("NOT FOUND","ERROR"): return None
    m=CUR.search(raw.replace(",",""))
    try: return float(m.group(1)) if m else None
    except: return None
def generate_summary(results):
    total=0.0; nwt=0; sps=defaultdict(float); breakdown=[]; flagged=[]
    for r in results:
        a=_parse(r.total_amount.value)
        store=r.store_name.value if r.store_name.value not in ("UNKNOWN","ERROR") else "Unknown Store"
        if a: total+=a; nwt+=1; sps[store]+=a
        breakdown.append({"file":r.file,"store":store,"date":r.date.value,
            "total":r.total_amount.value,"parsed_amount":a,
            "num_items":len(r.items),"overall_confidence":r.overall_confidence})
        if r.overall_confidence<0.70: flagged.append(r.file)
    return {"total_spend":round(total,2),"num_transactions":len(results),
            "num_with_total":nwt,"spend_per_store":{k:round(v,2) for k,v in sorted(sps.items(),key=lambda x:-x[1])},
            "receipts":breakdown,"flagged_receipts":flagged}
'''

with open('/content/ai_ocr/src/ocr_pipeline.py','w') as f: f.write(ocr_pipeline_src)
with open('/content/ai_ocr/src/financial_summary.py','w') as f: f.write(summary_src)
with open('/content/ai_ocr/src/__init__.py','w') as f: f.write('')
print('✅ Source files written')

✅ Source files written


In [ ]:
# ── Cell 4: Run the OCR pipeline ──────────────────────────────────────────────
import sys, json
sys.path.insert(0, '/content/ai_ocr')
from src.ocr_pipeline import ReceiptOCRPipeline
from src.financial_summary import generate_summary

OUTPUT_DIR = '/content/ai_ocr/outputs'

print('Initialising EasyOCR (downloads ~300MB weights on first run, please wait)...')
pipeline = ReceiptOCRPipeline(engine='auto')

print(f'\nProcessing receipts from: {RECEIPTS_FOLDER}\n')
results = pipeline.process_directory(RECEIPTS_FOLDER)

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
for r in results:
    stem = os.path.splitext(r.file)[0]
    with open(f'{OUTPUT_DIR}/{stem}.json', 'w') as f:
        json.dump(r.to_dict(), f, indent=2)
    flag = '⚠' if r.overall_confidence < 0.70 else '✓'
    print(f'  {flag}  {r.file}  (conf={r.overall_confidence:.2f})')

print(f'\n✅ Done! {len(results)} receipt(s) processed.')

Initialising EasyOCR (downloads ~300MB weights on first run, please wait)...



Processing receipts from: /content/receipts/AI-OCR dataset

  Processing 0.jpg...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  Processing 1.jpg...
  Processing 10.jpg...
  Processing 11.jpg...
  Processing 12.jpg...
  Processing 13.jpg...
  Processing 14.jpg...
  Processing 15.jpg...
  Processing 16.jpg...
  Processing 17.jpg...
  Processing 18.jpg...
  Processing 19.jpg...
  Processing 2.jpg...
  Processing 20.png...
  Processing 21.jpg...
  Processing 3.jpg...
  Processing 4.jpg...
  Processing 5.jpg...
  Processing 6.JPG...
  Processing 7.jpg...
  Processing 8.jpg...
  Processing 9.jpg...
  Processing X00016469612.jpg...
  Processing X00016469619.jpg...
  Processing X00016469620.jpg...
  Processing X00016469622.jpg...
  Processing X00016469623.jpg...
  Processing X00016469669.jpg...
  Processing X00016469670.jpg...
  Processing X00016469671.jpg...
  Processing X00016469672.jpg...
  Processing X00016469676.jpg...
  Processing X51005200931.jpg...
  Processing X51005200938.jpg...
  Processing X51005230605.jpg...
  Processing X51005230616.jpg...
  Processing X51005230617.jpg...
  Processing X51005230621.jpg..

In [ ]:
# ── Cell 5: Financial Summary ─────────────────────────────────────────────────
summary = generate_summary(results)
with open(f'{OUTPUT_DIR}/financial_summary.json','w') as f:
    json.dump(summary, f, indent=2)

print('='*50)
print('  FINANCIAL SUMMARY')
print('='*50)
print(f"  Transactions   : {summary['num_transactions']}")
print(f"  Totals found   : {summary['num_with_total']}")
print(f"  Total spend    : {summary['total_spend']:.2f}")
print(f"  Low-conf flags : {len(summary['flagged_receipts'])}")
if summary['spend_per_store']:
    print('\n  Spend per store:')
    for store, amt in summary['spend_per_store'].items():
        print(f'    {store:<28}  {amt:>10.2f}')
print(f'\n✅ Summary saved to {OUTPUT_DIR}/financial_summary.json')

In [ ]:
# ── Cell 6: Preview first result ──────────────────────────────────────────────
if results:
    print(json.dumps(results[0].to_dict(), indent=2))

In [ ]:
# ── Cell 7: Download all outputs as ZIP ───────────────────────────────────────
import shutil
shutil.make_archive('/content/ocr_outputs', 'zip', OUTPUT_DIR)
from google.colab import files
files.download('/content/ocr_outputs.zip')
print('✅ Download started — check your browser downloads')